In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import HTML
from google.colab import userdata
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Interrupt
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

def print_interrupts(interrupts: List[Interrupt]):
    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [ ]:
@tool
def send_email(recipient_email: str, customer_name: str, offer: str) -> str:
    """Call this tool to send a promotional email to a customer."""
    return f"Sent promo email to {customer_name} <{recipient_email}> about: {offer}"

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low"),
    tools=[send_email],
    system_prompt=(
        "You are a careful marketing assistant. "
        "Use tools only when the user clearly asks for a real-world action."
    ),
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                send_email.name: True
            }
        )
    ],
)

In [ ]:
# NOTE: The `result` will include an array of interrupts against the `__interrupt__` key. For each one of them, we must provide a decision in order to continue.
result = agent.invoke(
    input={
        "messages": [HumanMessage("Send a friendly promotional email to Maria at maria@example.com about a weekend-only 20% sneaker sale. Mention that the code is SPRING20.")]
    },
    config={
        "configurable": {
            "thread_id": "promotional_email_1"
        }
    }
)

In [ ]:
print_conversation(result["messages"])
print_interrupts(result.get("__interrupt__", []))

In [ ]:
# NOTE: We need to pass the same `thread_id` value - we are continuing the same thread.
# If we decide to reject instead, a "User rejected..." tool call result will be returned to the AI model.
continuation = agent.invoke(
    input=Command(resume={ "decisions": [{ "type": "approve" }] }),
    config={
        "configurable": {
            "thread_id": "promotional_email_1"
        }
    }
)

In [ ]:
print_conversation(continuation["messages"])